# Qwen14B Interview Seed Viewer

This notebook loads the generated interview conversation seeds and gives you a simple widget interface to:
- filter by scenario or text
- step through matching conversations
- inspect each message in order
- optionally inspect metadata

The notebook is preconfigured for:
`/playpen-ssd/smerrill/deception2/Interview/Data/Qwen14B/interview_conversation_seeds.jsonl`

In [ ]:
import html
import json
from pathlib import Path

import ipywidgets as widgets
import pandas as pd
from IPython.display import HTML, Markdown, display

DATA_PATH = Path('/playpen-ssd/smerrill/deception2/Interview/Data/Qwen14B/interview_conversation_seeds.jsonl')
print('data_path =', DATA_PATH)
print('exists =', DATA_PATH.exists())

In [ ]:
def load_records(path: Path):
    records = []
    with path.open('r', encoding='utf-8') as handle:
        for line_idx, line in enumerate(handle, start=1):
            text = line.strip()
            if not text:
                continue
            row = json.loads(text)
            row['_line_idx'] = line_idx
            row['_flat_text'] = ' '.join(
                str(turn.get('message', '')) for turn in row.get('seeded_dialogue', [])
            ).lower()
            records.append(row)
    return records


records = load_records(DATA_PATH)
summary_rows = []
for idx, row in enumerate(records):
    meta = row.get('metadata', {}) or {}
    summary_rows.append(
        {
            'record_idx': idx,
            'line_idx': row.get('_line_idx'),
            'conversation_id': row.get('conversation_id'),
            'scenario': row.get('base_scenario_name'),
            'turns': len(row.get('seeded_dialogue', [])),
            'global_idx': meta.get('global_idx'),
            'shard_index': meta.get('shard_index'),
        }
    )

summary_df = pd.DataFrame(summary_rows)
scenario_counts = summary_df['scenario'].value_counts().rename_axis('scenario').reset_index(name='count')

print('num_records =', len(records))
display(scenario_counts)
display(summary_df.head(10))

In [ ]:
scenario_options = ['All'] + sorted(summary_df['scenario'].dropna().unique().tolist())

scenario_dropdown = widgets.Dropdown(
    options=scenario_options,
    value='All',
    description='Scenario:',
    layout=widgets.Layout(width='320px'),
)
text_filter = widgets.Text(
    value='',
    description='Filter:',
    placeholder='conversation_id or message text',
    layout=widgets.Layout(width='420px'),
)
record_slider = widgets.IntSlider(
    value=0,
    min=0,
    max=max(len(records) - 1, 0),
    step=1,
    description='Match #',
    continuous_update=False,
    layout=widgets.Layout(width='420px'),
)
prev_button = widgets.Button(description='Previous', button_style='')
next_button = widgets.Button(description='Next', button_style='')
show_metadata = widgets.Checkbox(value=False, description='Show metadata')

stats_html = widgets.HTML()
selected_html = widgets.HTML()
output = widgets.Output()


def matching_record_indices():
    scenario_value = scenario_dropdown.value
    text_value = text_filter.value.strip().lower()
    matches = []
    for record_idx, row in enumerate(records):
        if scenario_value != 'All' and row.get('base_scenario_name') != scenario_value:
            continue
        if text_value:
            convo_id = str(row.get('conversation_id', '')).lower()
            scenario_name = str(row.get('base_scenario_name', '')).lower()
            if (
                text_value not in convo_id
                and text_value not in scenario_name
                and text_value not in row.get('_flat_text', '')
            ):
                continue
        matches.append(record_idx)
    return matches


def render_record(record):
    meta = record.get('metadata', {}) or {}
    parts = []
    parts.append(
        '<div style="margin: 8px 0 16px 0; padding: 10px 12px; border: 1px solid #d1d5db; border-radius: 10px; background: #f8fafc;">'
        f"<div><b>Conversation ID:</b> {html.escape(str(record.get('conversation_id', '')))}</div>"
        f"<div><b>Scenario:</b> {html.escape(str(record.get('base_scenario_name', '')))}</div>"
        f"<div><b>Turns:</b> {len(record.get('seeded_dialogue', []))}</div>"
        '</div>'
    )

    for turn_idx, turn in enumerate(record.get('seeded_dialogue', []), start=1):
        speaker = str(turn.get('speaker', ''))
        message = html.escape(str(turn.get('message', ''))).replace('\n', '<br>')
        is_candidate = speaker.strip().lower() == 'candidate'
        border = '#2563eb' if is_candidate else '#059669'
        bg = '#eff6ff' if is_candidate else '#ecfdf5'
        parts.append(
            f'<div style="margin: 10px 0; padding: 12px 14px; border-left: 5px solid {border}; '
            f'background: {bg}; border-radius: 8px;">'
            f'<div style="font-weight: 700; margin-bottom: 6px;">Turn {turn_idx}: {html.escape(speaker)}</div>'
            f'<div style="line-height: 1.5; white-space: normal;">{message}</div>'
            '</div>'
        )

    if show_metadata.value:
        parts.append(
            '<div style="margin-top: 16px; padding: 12px 14px; border: 1px solid #e5e7eb; border-radius: 8px; background: #fafafa;">'
            '<div style="font-weight: 700; margin-bottom: 8px;">Metadata</div>'
            f'<pre style="margin: 0; white-space: pre-wrap;">{html.escape(json.dumps(meta, indent=2, ensure_ascii=False))}</pre>'
            '</div>'
        )

    return HTML(''.join(parts))


def refresh(*_):
    matches = matching_record_indices()
    record_slider.max = max(len(matches) - 1, 0)
    record_slider.disabled = len(matches) == 0
    prev_button.disabled = len(matches) == 0 or record_slider.value <= 0
    next_button.disabled = len(matches) == 0 or record_slider.value >= max(len(matches) - 1, 0)
    if record_slider.value > record_slider.max:
        record_slider.value = record_slider.max

    if not matches:
        stats_html.value = '<b>Matches:</b> 0'
        selected_html.value = '<b>Selected:</b> none'
        with output:
            output.clear_output()
            display(Markdown('No matching conversations.'))
        return

    selected_record = records[matches[record_slider.value]]
    stats_html.value = f'<b>Matches:</b> {len(matches)}'
    selected_html.value = (
        f"<b>Selected:</b> {html.escape(str(selected_record.get('conversation_id', '')))} "
        f"(file line {selected_record.get('_line_idx')})"
    )
    with output:
        output.clear_output()
        display(render_record(selected_record))


def on_prev(_):
    if record_slider.value > 0:
        record_slider.value -= 1


def on_next(_):
    if record_slider.value < record_slider.max:
        record_slider.value += 1


scenario_dropdown.observe(refresh, names='value')
text_filter.observe(refresh, names='value')
record_slider.observe(refresh, names='value')
show_metadata.observe(refresh, names='value')
prev_button.on_click(on_prev)
next_button.on_click(on_next)

controls_top = widgets.HBox([scenario_dropdown, text_filter])
controls_bottom = widgets.HBox([prev_button, next_button, record_slider, show_metadata])
header = widgets.VBox([controls_top, controls_bottom, stats_html, selected_html])

display(header)
display(output)
refresh()